# Lab 8.4 &mdash; Blast Radius and Tool Governance

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 3 &middot; Module 8 &mdash; Safety &amp; Guardrails**

### What you'll do
- Classify every tool: read, reversible write, or irreversible
- Compute blast radius &mdash; what an attacker gets if the agent is fully theirs
- Shrink it with least privilege, and see what actually breaks
- Produce the grant a reviewer can approve in a minute

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **Stop asking whether it is safe.** That question has no answer. Ask what it can do,
> which is a list you can shorten.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-8-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the tools, and what they can do
# Ten tools an operations agent might plausibly be granted. Note the middle group:
# writes you can undo. Most governance conversations only have two boxes.

TOOLS = {
    "lookup_payment":   {"writes": False, "reversible": True,  "external": False,
                         "scope": "one payment"},
    "search_payments":  {"writes": False, "reversible": True,  "external": False,
                         "scope": "the whole book"},
    "policy_for":       {"writes": False, "reversible": True,  "external": False,
                         "scope": "public runbooks"},
    "retrieve":         {"writes": False, "reversible": True,  "external": False,
                         "scope": "the index"},
    "open_ticket":      {"writes": True,  "reversible": True,  "external": False,
                         "scope": "case system"},
    "add_case_note":    {"writes": True,  "reversible": True,  "external": False,
                         "scope": "case system"},
    "draft_email":      {"writes": True,  "reversible": True,  "external": False,
                         "scope": "drafts folder"},
    "send_email":       {"writes": True,  "reversible": False, "external": True,
                         "scope": "anyone"},
    "release_payment":  {"writes": True,  "reversible": False, "external": True,
                         "scope": "the payments book"},
    "purge_case":       {"writes": True,  "reversible": False, "external": False,
                         "scope": "case system"},
}

print(f"{len(TOOLS)} tools to classify")

## Concept

&ldquo;Is this agent secure?&rdquo; is unanswerable and every review stalls on it. Replace it:

> **If this agent were entirely under an attacker's control, what could they do?**

That has a concrete answer &mdash; the tools you granted, and the scope of each. It is a list, and a
list can be shortened. Nothing about the model enters into it.

## Section 1 &mdash; Classify

Three classes, and the middle one is the one most governance frameworks do not have.

In [ ]:
def classify(tool: str) -> str:
    """read | reversible write | irreversible."""
    t = TOOLS[tool]
    if not t["writes"]:
        return "read"
    return "reversible write" if t["reversible"] else "irreversible"


def unattended_ok(tool: str) -> bool:
    """May the agent call this without a human in the loop?"""
    return classify(tool) != "irreversible"


def by_class() -> dict:
    out = {}
    for name in TOOLS:
        out.setdefault(classify(name), []).append(name)
    return out

In [ ]:
# --- Self-check: Section 1
check("every tool lands in exactly one class",
      lambda: sum(len(v) for v in by_class().values()) == len(TOOLS))
check("there are three classes, not two",
      lambda: set(by_class()) == {"read", "reversible write", "irreversible"},
      "the middle class is the one most policies forget, and it is where most tools live")
check("releasing a payment is irreversible",
      lambda: classify("release_payment") == "irreversible")
check("drafting an email is a reversible write; sending one is not",
      lambda: classify("draft_email") == "reversible write"
              and classify("send_email") == "irreversible",
      "the same verb, one step apart, and a completely different control")
check("only the irreversible tools are barred from running unattended",
      lambda: [t for t in TOOLS if not unattended_ok(t)] == sorted(by_class()["irreversible"]) or
              set(t for t in TOOLS if not unattended_ok(t)) == set(by_class()["irreversible"]))
check("the irreversible list is short, and deliberately so",
      lambda: len(by_class()["irreversible"]) <= 3)

def _classes():
    for k in ("read", "reversible write", "irreversible"):
        print(f"  {k:18} {', '.join(sorted(by_class()[k]))}")
guard(_classes)

## Section 2 &mdash; The blast radius

Given a grant, what does an attacker get? Score it so two designs can be compared, and so a
change to the grant shows up as a number.

In [ ]:
GENEROUS = set(TOOLS)                                     # everything, unattended
LEAST_PRIVILEGE = {"lookup_payment", "policy_for", "retrieve", "draft_email", "add_case_note"}

WEIGHT = {"read": 1, "reversible write": 3, "irreversible": 10}

def blast_radius(grant: set, gated: set = frozenset()) -> dict:
    """What an attacker controlling this agent could do.

    `gated` names tools that need a named human, so an attacker cannot reach them alone.
    """
    reachable = [t for t in grant if t not in gated]
    score = sum(WEIGHT[classify(t)] for t in reachable)
    return {"score": score,
            "reachable": len(reachable),
            "irreversible": sorted(t for t in reachable if classify(t) == "irreversible"),
            "external": sorted(t for t in reachable if TOOLS[t]["external"])}

In [ ]:
# --- Self-check: Section 2
def irreversible_tools() -> set:
    """Computed on demand. A module-level call into classify() -- which has a blank in it --
    would raise NameError when the CELL runs, crashing it instead of printing [TODO]."""
    return {t for t in TOOLS if classify(t) == "irreversible"}

check("granting everything gives the largest radius",
      lambda: blast_radius(GENEROUS)["score"] > blast_radius(LEAST_PRIVILEGE)["score"])
check("and it reaches every irreversible tool",
      lambda: set(blast_radius(GENEROUS)["irreversible"]) == irreversible_tools())
check("least privilege reaches none of them",
      lambda: blast_radius(LEAST_PRIVILEGE)["irreversible"] == [])
check("GATING IS AS STRONG AS NOT GRANTING, for the irreversible ones",
      lambda: blast_radius(GENEROUS, gated=irreversible_tools())["irreversible"] == [],
      "the agent may still call them; an attacker alone cannot complete one")
check("but gating leaves more reachable overall",
      lambda: blast_radius(GENEROUS, gated=irreversible_tools())["score"]
              > blast_radius(LEAST_PRIVILEGE)["score"],
      "a gate is not a substitute for not granting a tool you never needed")
check("nothing external survives least privilege",
      lambda: blast_radius(LEAST_PRIVILEGE)["external"] == [],
      "reaching outside the organisation is the step you cannot take back")
check("the score falls monotonically as you remove tools",
      lambda: blast_radius(LEAST_PRIVILEGE - {"draft_email"})["score"]
              < blast_radius(LEAST_PRIVILEGE)["score"])

def _radius():
    for label, grant, gated in (("everything, ungated", GENEROUS, frozenset()),
                                ("everything, gated  ", GENEROUS, irreversible_tools()),
                                ("least privilege    ", LEAST_PRIVILEGE, frozenset())):
        r = blast_radius(grant, gated)
        print(f"  {label}  score {r['score']:>3}  irreversible reachable: "
              f"{r['irreversible'] or 'none'}")
guard(_radius)

## Section 3 &mdash; What breaks when you shrink it

Least privilege is only a real proposal if you know what it costs. Run the workload and find out
which tasks stop working.

In [ ]:
TASKS = {
    "explain a failure":        {"lookup_payment", "policy_for"},
    "find related payments":    {"search_payments"},
    "answer from the runbook":  {"retrieve", "policy_for"},
    "record the decision":      {"add_case_note"},
    "prepare a client note":    {"draft_email"},
    "notify the client":        {"send_email"},
    "release the payment":      {"release_payment"},
}

def supported(task: str, grant: set) -> bool:
    """Can this task run with this grant?"""
    return TASKS[task] <= grant


def coverage(grant: set) -> dict:
    ok = [t for t in TASKS if supported(t, grant)]
    return {"supported": sorted(ok),
            "blocked": sorted(t for t in TASKS if t not in ok),
            "rate": len(ok) / len(TASKS)}

In [ ]:
# --- Self-check: Section 3
check("the generous grant supports everything",
      lambda: coverage(GENEROUS)["rate"] == 1.0)
check("least privilege still supports the majority of the work",
      lambda: coverage(LEAST_PRIVILEGE)["rate"] > 0.5,
      "four tasks out of seven, having removed every irreversible tool -- less than people fear")
check("what it blocks is exactly the irreversible work",
      lambda: set(coverage(LEAST_PRIVILEGE)["blocked"])
              == {"find related payments", "notify the client", "release the payment"})
check("two of those three are irreversible; the third is a scope question",
      lambda: "find related payments" in coverage(LEAST_PRIVILEGE)["blocked"]
              and TOOLS["search_payments"]["scope"] == "the whole book",
      "search reads nothing dangerous -- it just reads EVERYTHING, which is its own problem")
check("adding search back costs one point of radius and unblocks a task",
      lambda: coverage(LEAST_PRIVILEGE | {"search_payments"})["rate"]
              > coverage(LEAST_PRIVILEGE)["rate"]
          and blast_radius(LEAST_PRIVILEGE | {"search_payments"})["score"]
              == blast_radius(LEAST_PRIVILEGE)["score"] + 1)
check("gating release rather than removing it restores that task with a human in it",
      lambda: supported("release the payment", GENEROUS) is True)

def _coverage():
    for label, grant in (("everything", GENEROUS), ("least privilege", LEAST_PRIVILEGE)):
        c = coverage(grant)
        print(f"  {label:16} {c['rate']:.0%} of tasks   blocked: {c['blocked']}")
guard(_coverage)

## Section 4 &mdash; The grant a reviewer can approve

One table. A reviewer should be able to read it in a minute and say yes or no.

In [ ]:
def proposed_grant() -> dict:
    """Least privilege, plus the irreversible tools behind a gate."""
    grant = LEAST_PRIVILEGE | {"search_payments"} | irreversible_tools()
    gated = irreversible_tools()
    r = blast_radius(grant, gated)
    c = coverage(grant)
    return {"grant": sorted(grant), "gated": sorted(gated),
            "task_coverage": c["rate"], "blast_radius": r["score"],
            "unattended_irreversible": r["irreversible"]}


def review_table() -> list:
    g = proposed_grant()
    return [{"tool": t, "class": classify(t), "scope": TOOLS[t]["scope"],
             "unattended": t not in g["gated"]} for t in g["grant"]]

In [ ]:
# --- Self-check: Section 4
check("the proposal covers every task",
      lambda: proposed_grant()["task_coverage"] == 1.0,
      "you do not have to give up capability to remove unattended risk")
check("and no irreversible tool is reachable unattended",
      lambda: proposed_grant()["unattended_irreversible"] == [])
check("the review table has a row per granted tool",
      lambda: len(review_table()) == len(proposed_grant()["grant"]))
check("every row states a class and a scope",
      lambda: all(r["class"] and r["scope"] for r in review_table()))
check("exactly the irreversible rows are marked as needing a human",
      lambda: {r["tool"] for r in review_table() if not r["unattended"]} == irreversible_tools())

def _review():
    g = proposed_grant()
    print(f"  task coverage {g['task_coverage']:.0%}   blast radius {g['blast_radius']}"
          f"   unattended irreversible: {g['unattended_irreversible'] or 'none'}\n")
    print(f"  {'tool':18}{'class':20}{'scope':22}{'unattended'}")
    print("  " + "-" * 68)
    for r in review_table():
        print(f"  {r['tool']:18}{r['class']:20}{r['scope']:22}{'yes' if r['unattended'] else 'NO'}")
guard(_review)

## Run it for real

Give the model the tool list and ask it to propose a grant. Then compare its answer with yours &mdash;
not to grade the model, but because a governance conversation with a starting draft goes faster.

In [ ]:
if llm_ready():
    def _propose():
        listing = "\n".join(
            f"- {t}: writes={v['writes']}, reversible={v['reversible']}, "
            f"external={v['external']}, scope={v['scope']}" for t, v in TOOLS.items())
        reply = ask("An operations agent investigates failed payments and recommends an action. "
                    "From these tools, say which it should be granted, which must require human "
                    "approval, and which it should not have at all. Be brief.\n\n" + listing)
        print(reply.strip()[:600])
        print(f"\n  your proposal: unattended-irreversible = "
              f"{proposed_grant()['unattended_irreversible'] or 'none'}, "
              f"coverage {proposed_grant()['task_coverage']:.0%}")
    guard(_propose)

### Read it

A model is usually good at this, because it is a classification task with visible features, and a
sensible draft grant in thirty seconds is worth having.

It is still a draft. The model does not know that your `search_payments` reads the whole book, that
your drafts folder is shared with a team, or that `purge_case` is used by an overnight job that
would break. Those facts live with people, and the table you produced in Section 4 is the artefact
that gets them into the room.

In [ ]:
score()

## Your turn

1. `WEIGHT` says an irreversible tool is worth ten reads. Defend or change those numbers. What
   would make a read genuinely worse than a reversible write? (`search_payments` is a hint.)
2. Add a `rate_limit` to the reversible-write class and decide the number for `add_case_note`.
   What does an attacker do with a thousand case notes?
3. Blast radius here counts tools. Extend it to count *data*: a read scoped to one payment and a
   read scoped to the whole book are both class `read` and are not the same risk.